In [15]:
import pandas as pd

In [16]:
df = pd.read_csv("data/conexao_politica_articles.csv", sep="|")
df.head()

,id,title,content
0,CNXP-opiniao-20250712-a1983d80,Lula minimiza papel do governo na alta do gás ...,O presidente Luiz Inácio Lula da Silva (PT) mi...
1,CNXP-opiniao-20250607-61b5a608,Quer ser o novo Nikolas Ferreira?,Hoje eu vou te ensinar como ser o novo Nikolas...
2,CNXP-opiniao-20250512-d13c5d51,Ser mãe segundo o coração de Deus: mais do que...,Em uma época de tantos discursos e pouca coerê...
3,CNXP-opiniao-20250502-7491424b,"100 dias de mandato: Trump, tarifas e a arte d...",Enquanto você reclama da falta de engajamento ...
4,CNXP-opiniao-20241126-a4eb0f58,Conservador que chamou petista de &#8216;ladrã...,O ex-deputado estadual e vereador eleito de Ca...


In [17]:
text = df.iloc[0]["content"]
# change all repeated \n to a single \n
text = "\n".join([line.strip() for line in text.split("\n") if line.strip()])
print(text)


O presidente Luiz Inácio Lula da Silva (PT) minimizou nesta sexta-feira (11) a responsabilidade do governo sobre a alta no preço do gás de cozinha nos últimos meses. Em nova fala sobre o assunto, ele criticou a diferença entre o valor praticado pela Petrobras e o preço final cobrado das famílias brasileiras pelas distribuidoras.
Durante cerimônia no Espírito Santo, Lula afirmou que ainda não recebeu uma justificativa convincente sobre o encarecimento do produto ao longo da cadeia de distribuição.
“Eu não me conformo que o gás de cozinha sai da Petrobras a R$ 37 e chega a R$ 140 nas casas dos consumidores. Alguém tem que me explicar”, disse o presidente em evento em Linhares (ES), durante o lançamento dos pagamentos do Programa de Transferência de Renda (PTR) destinado a pescadores e agricultores familiares afetados pelo rompimento da barragem de Fundão, em Mariana (MG).
De acordo com dados da Petrobras, o preço médio do gás liquefeito de petróleo (GLP) nas refinarias está atualmente em

In [18]:
from nltk.tokenize import PunktTokenizer

sent_detector = PunktTokenizer("portuguese")
sentences = sent_detector.tokenize(text)
for sentence in sentences:
    print(sentence)

O presidente Luiz Inácio Lula da Silva (PT) minimizou nesta sexta-feira (11) a responsabilidade do governo sobre a alta no preço do gás de cozinha nos últimos meses.
Em nova fala sobre o assunto, ele criticou a diferença entre o valor praticado pela Petrobras e o preço final cobrado das famílias brasileiras pelas distribuidoras.
Durante cerimônia no Espírito Santo, Lula afirmou que ainda não recebeu uma justificativa convincente sobre o encarecimento do produto ao longo da cadeia de distribuição.
“Eu não me conformo que o gás de cozinha sai da Petrobras a R$ 37 e chega a R$ 140 nas casas dos consumidores.
Alguém tem que me explicar”, disse o presidente em evento em Linhares (ES), durante o lançamento dos pagamentos do Programa de Transferência de Renda (PTR) destinado a pescadores e agricultores familiares afetados pelo rompimento da barragem de Fundão, em Mariana (MG).
De acordo com dados da Petrobras, o preço médio do gás liquefeito de petróleo (GLP) nas refinarias está atualmente em

In [19]:
# remove cases in which we have just plain \n\n\n\n or more, to only have one \n between sentences
df["content"] = df["content"].str.replace(r'\n+', '\n', regex=True)

In [20]:
# save a new sentences csv by saving
# id, sentence

df["sentences"] = df["content"].apply(lambda x: sent_detector.tokenize(x))
sentences_df = df[["id", "sentences"]].explode("sentences")
sentences_df = sentences_df.rename(columns={"sentences": "sentence"})
sentences_df = sentences_df.reset_index(drop=True)
sentences_df

,id,sentence
0,CNXP-opiniao-20250712-a1983d80,O presidente Luiz Inácio Lula da Silva (PT) mi...
1,CNXP-opiniao-20250712-a1983d80,"Em nova fala sobre o assunto, ele criticou a d..."
2,CNXP-opiniao-20250712-a1983d80,"Durante cerimônia no Espírito Santo, Lula afir..."
3,CNXP-opiniao-20250712-a1983d80,“Eu não me conformo que o gás de cozinha sai d...
4,CNXP-opiniao-20250712-a1983d80,"Alguém tem que me explicar”, disse o president..."
...,...,...
20113,CNXP-analise-20180212-0492aafd,Se você se ofendeu... já sabe o que penso de ti.
20114,CNXP-analise-20180212-0492aafd,Se não se ofendeu: continue a ler.
20115,CNXP-analise-20180212-0492aafd,"Se tem algo de belo no carnaval, creio que sej..."
20116,CNXP-analise-20180212-0492aafd,Texto de Anderson C. Sandes&nbsp;\nNotas:\n[1]...


In [21]:
# remove all sentence occurrences in with we have less than 3 words
print(sentences_df.shape)
sentences_df = sentences_df[sentences_df["sentence"].str.split().apply(len) >= 3]
# remove all occurrences including "nbsp"
sentences_df = sentences_df[~sentences_df["sentence"].str.contains("nbsp")]
# remove all occurrences including "reportagem" + "App" (lowered)
sentences_df = sentences_df[~sentences_df["sentence"].str.contains("reportagem", case=False) & ~sentences_df["sentence"].str.contains("app", case=False)]
# remove all "nosso Aplicativo" (lowered)
sentences_df = sentences_df[~sentences_df["sentence"].str.contains("nosso aplicativo", case=False)]
# remove all "conexão política" + loja (lowered)
sentences_df = sentences_df[~sentences_df["sentence"].str.contains("conexão política", case=False) & ~sentences_df["sentence"].str.contains("loja", case=False)]
# remove all "É 100% gratuito" (lowered)
sentences_df = sentences_df[~sentences_df["sentence"].str.contains("é 100% gratuito", case=False)]

print(sentences_df.shape)
sentences_df.head()

(20118, 2)
(18364, 2)


,id,sentence
0,CNXP-opiniao-20250712-a1983d80,O presidente Luiz Inácio Lula da Silva (PT) mi...
1,CNXP-opiniao-20250712-a1983d80,"Em nova fala sobre o assunto, ele criticou a d..."
2,CNXP-opiniao-20250712-a1983d80,"Durante cerimônia no Espírito Santo, Lula afir..."
3,CNXP-opiniao-20250712-a1983d80,“Eu não me conformo que o gás de cozinha sai d...
4,CNXP-opiniao-20250712-a1983d80,"Alguém tem que me explicar”, disse o president..."


In [22]:
# after exploding, put an index into id so we dont have duplicate ids
# so we will have f"{id}-{index}" as the new id
sentences_df["id"] = sentences_df.groupby("id").cumcount().astype(str).radd(sentences_df["id"].astype(str) + "-")
sentences_df.head()

,id,sentence
0,CNXP-opiniao-20250712-a1983d80-0,O presidente Luiz Inácio Lula da Silva (PT) mi...
1,CNXP-opiniao-20250712-a1983d80-1,"Em nova fala sobre o assunto, ele criticou a d..."
2,CNXP-opiniao-20250712-a1983d80-2,"Durante cerimônia no Espírito Santo, Lula afir..."
3,CNXP-opiniao-20250712-a1983d80-3,“Eu não me conformo que o gás de cozinha sai d...
4,CNXP-opiniao-20250712-a1983d80-4,"Alguém tem que me explicar”, disse o president..."


In [23]:
# iter by df and try to find all "texto de" occurrences
texto_de_counts = 0

for index, row in df.iterrows():
    text = row["content"]
    sentences = sent_detector.tokenize(text)
    for sentence in sentences:
        if "texto de" in sentence.lower():
            if texto_de_counts < 5:
                print(f"Found 'texto de' in article {row['id']}: {sentence}")
            texto_de_counts += 1

print(f"Total occurrences of 'texto de': {texto_de_counts}")

Found 'texto de' in article CNXP-opiniao-20210305-f8038d40: Mesmo após a desconstrução de inúmeras falácias criadas durante o contexto de pandemia, nossos governantes continuam a insistir nos mesmos erros ante ao enfrentamento do vírus, principalmente no tocante às medidas restritivas de liberdade — os famosos lockdowns — adotadas por toda a nação.
Found 'texto de' in article CNXP-opiniao-20210102-33643cae: Essa nova medida visa alterar nossos padrões de comportamento e convivência, sob o pretexto de que nossa sobrevivência e proteção estão ameaçados pelo covid-19.
Found 'texto de' in article CNXP-opiniao-20201124-37c32bf1: Caso você tenha perdido, em 18 de novembro, o Parlamento alemão aprovou uma lei, o chamado "Ato de Proteção contra Infecções" ("Das Infektionsschutzgesetz", em alemão), concedendo formalmente ao governo a autoridade para emitir qualquer decreto que quiser sob o pretexto de ‘proteger a saúde pública’.
Found 'texto de' in article CNXP-opiniao-20201124-37c32bf1: Agora,

In [24]:
# tokenize text from sentences_df
from nltk.tokenize import word_tokenize
sentences_df["tokens"] = sentences_df["sentence"].apply(word_tokenize)
sentences_df.head()

,id,sentence,tokens
0,CNXP-opiniao-20250712-a1983d80-0,O presidente Luiz Inácio Lula da Silva (PT) mi...,"[O, presidente, Luiz, Inácio, Lula, da, Silva,..."
1,CNXP-opiniao-20250712-a1983d80-1,"Em nova fala sobre o assunto, ele criticou a d...","[Em, nova, fala, sobre, o, assunto, ,, ele, cr..."
2,CNXP-opiniao-20250712-a1983d80-2,"Durante cerimônia no Espírito Santo, Lula afir...","[Durante, cerimônia, no, Espírito, Santo, ,, L..."
3,CNXP-opiniao-20250712-a1983d80-3,“Eu não me conformo que o gás de cozinha sai d...,"[“, Eu, não, me, conformo, que, o, gás, de, co..."
4,CNXP-opiniao-20250712-a1983d80-4,"Alguém tem que me explicar”, disse o president...","[Alguém, tem, que, me, explicar, ”, ,, disse, ..."


In [25]:
# remove stop words and ponctuation to new "tokens_no_stop" column
from nltk.corpus import stopwords
import string

stop_words = set(stopwords.words("portuguese"))
special_chars = ["nbsp", "nbsp;", "nbsp", "nbsp;", "\n", "\t", "'", '"', "`", "``", "''", "“", "”", "‘", "’"]

sentences_df["tokens_no_stop"] = sentences_df["tokens"].apply(
    lambda tokens: [
        token for token in tokens 
        if (token.lower() not in stop_words)
        and (token not in string.punctuation) 
        and (token.lower() not in special_chars)
    ]
)
sentences_df.head()

,id,sentence,tokens,tokens_no_stop
0,CNXP-opiniao-20250712-a1983d80-0,O presidente Luiz Inácio Lula da Silva (PT) mi...,"[O, presidente, Luiz, Inácio, Lula, da, Silva,...","[presidente, Luiz, Inácio, Lula, Silva, PT, mi..."
1,CNXP-opiniao-20250712-a1983d80-1,"Em nova fala sobre o assunto, ele criticou a d...","[Em, nova, fala, sobre, o, assunto, ,, ele, cr...","[nova, fala, sobre, assunto, criticou, diferen..."
2,CNXP-opiniao-20250712-a1983d80-2,"Durante cerimônia no Espírito Santo, Lula afir...","[Durante, cerimônia, no, Espírito, Santo, ,, L...","[Durante, cerimônia, Espírito, Santo, Lula, af..."
3,CNXP-opiniao-20250712-a1983d80-3,“Eu não me conformo que o gás de cozinha sai d...,"[“, Eu, não, me, conformo, que, o, gás, de, co...","[conformo, gás, cozinha, sai, Petrobras, R, 37..."
4,CNXP-opiniao-20250712-a1983d80-4,"Alguém tem que me explicar”, disse o president...","[Alguém, tem, que, me, explicar, ”, ,, disse, ...","[Alguém, explicar, disse, presidente, evento, ..."


In [26]:
# analyze bigram in each sentence and count the most common bigrams
from collections import Counter
from nltk import bigrams

bigram_counts = Counter()
for sentence in sentences_df["sentence"]:
    bigram_counts.update(bigrams(sentence.split()))

display(bigram_counts.most_common(20))

[(('que', 'o'), 946),
 (('o', 'que'), 728),
 (('que', 'a'), 710),
 (('que', 'não'), 565),
 (('com', 'a'), 556),
 (('com', 'o'), 551),
 (('para', 'o'), 504),
 (('e', 'a'), 502),
 (('de', 'um'), 496),
 (('para', 'a'), 494),
 (('e', 'o'), 445),
 (('de', 'uma'), 411),
 (('de', 'que'), 371),
 (('é', 'o'), 370),
 (('não', 'é'), 346),
 (('que', 'se'), 346),
 (('do', 'que'), 319),
 (('que', 'os'), 315),
 (('que', 'é'), 287),
 (('para', 'que'), 283)]

In [27]:
# bigram counts for tokens_no_stop
bigram_counts_no_stop = Counter()
for tokens in sentences_df["tokens_no_stop"]:
    bigram_counts_no_stop.update(bigrams(tokens))

display(bigram_counts_no_stop.most_common(20))

[(('Jair', 'Bolsonaro'), 217),
 (('redes', 'sociais'), 149),
 (('Estados', 'Unidos'), 115),
 (('cada', 'vez'), 83),
 (('Além', 'disso'), 82),
 (('Donald', 'Trump'), 75),
 (('presidente', 'Jair'), 75),
 (('muitas', 'vezes'), 67),
 (('Boko', 'Haram'), 65),
 (('Rio', 'Janeiro'), 59),
 (('liberdade', 'expressão'), 55),
 (('Partido', 'Comunista'), 54),
 (('todo', 'mundo'), 53),
 (('Congresso', 'Nacional'), 53),
 (('Supremo', 'Tribunal'), 50),
 (('Lava', 'Jato'), 47),
 (('outro', 'lado'), 46),
 (('Tribunal', 'Federal'), 45),
 (('Hong', 'Kong'), 44),
 (('ano', 'passado'), 40)]

In [28]:
display(sentences_df.head())
display(sentences_df.shape)

sentences_df[['id', 'sentence']].to_csv("data/conexao_politica_sentences.csv", index=False, sep="|")

,id,sentence,tokens,tokens_no_stop
0,CNXP-opiniao-20250712-a1983d80-0,O presidente Luiz Inácio Lula da Silva (PT) mi...,"[O, presidente, Luiz, Inácio, Lula, da, Silva,...","[presidente, Luiz, Inácio, Lula, Silva, PT, mi..."
1,CNXP-opiniao-20250712-a1983d80-1,"Em nova fala sobre o assunto, ele criticou a d...","[Em, nova, fala, sobre, o, assunto, ,, ele, cr...","[nova, fala, sobre, assunto, criticou, diferen..."
2,CNXP-opiniao-20250712-a1983d80-2,"Durante cerimônia no Espírito Santo, Lula afir...","[Durante, cerimônia, no, Espírito, Santo, ,, L...","[Durante, cerimônia, Espírito, Santo, Lula, af..."
3,CNXP-opiniao-20250712-a1983d80-3,“Eu não me conformo que o gás de cozinha sai d...,"[“, Eu, não, me, conformo, que, o, gás, de, co...","[conformo, gás, cozinha, sai, Petrobras, R, 37..."
4,CNXP-opiniao-20250712-a1983d80-4,"Alguém tem que me explicar”, disse o president...","[Alguém, tem, que, me, explicar, ”, ,, disse, ...","[Alguém, explicar, disse, presidente, evento, ..."


(18364, 4)